In [ ]:
#%conda install -c conda-forge lightfm -y

2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.11.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /opt/homebrew/Caskroom/miniconda/base/envs/data_analysis

  added / updated specs:
    - lightfm


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    lightfm-1.17               |  py311h4add359_2         246 KB  conda-forge
    numpy-1.26.4               |  py311h901140f_1          14 KB
    numpy-base-1.26.4          |  py311hae06d03_1         6.9 MB
    ------------------------------------------------------------
                                           Total:         7.2 MB

The following NEW packages will be INSTAL

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix
from lightfm import LightFM
from lightfm.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
np.random.seed(42)

/opt/homebrew/Caskroom/miniconda/base/envs/data_analysis/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [3]:
df = pd.read_csv('../../data/children_products/children_product_cleaned_full_year.csv')
print(f"Размер исходного датасета: {df.shape}")

Размер исходного датасета: (4050261, 17)


In [4]:
df_filtered = df[(df['Статус'] == 'Доставлен') & (df['Отменено'] == 'Нет')].copy()
df_filtered = df_filtered.dropna(subset=['Телефон_new', 'ID_SKU', 'Дата'])
df_filtered['Дата'] = pd.to_datetime(df_filtered['Дата'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['Дата'])

print(f"После фильтрации (доставленные): {df_filtered.shape}")
print(f"Диапазон дат: {df_filtered['Дата'].min()} - {df_filtered['Дата'].max()}")

После фильтрации (доставленные): (2053127, 17)
Диапазон дат: 2017-01-01 17:38:00 - 2017-12-01 19:48:00


In [5]:
MIN_INTERACTIONS = 3

for _ in range(5):
    user_counts = df_filtered.groupby('Телефон_new').size()
    item_counts = df_filtered.groupby('ID_SKU').size()
    active_users = user_counts[user_counts >= MIN_INTERACTIONS].index
    active_items = item_counts[item_counts >= MIN_INTERACTIONS].index
    before = len(df_filtered)
    df_filtered = df_filtered[
        df_filtered['Телефон_new'].isin(active_users) &
        df_filtered['ID_SKU'].isin(active_items)
    ]
    if len(df_filtered) == before:
        break

n_users = df_filtered['Телефон_new'].nunique()
n_items = df_filtered['ID_SKU'].nunique()
print(f"После фильтрации (≥{MIN_INTERACTIONS} покупок):")
print(f"  Пользователей: {n_users:,}, Товаров: {n_items:,}")
print(f"  Взаимодействий: {len(df_filtered):,}")

После фильтрации (≥3 покупок):
  Пользователей: 129,421, Товаров: 71,502
  Взаимодействий: 1,694,548


In [6]:
interactions = df_filtered.groupby(['Телефон_new', 'ID_SKU']).size().reset_index(name='count')

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
interactions['user_id'] = user_encoder.fit_transform(interactions['Телефон_new'])
interactions['item_id'] = item_encoder.fit_transform(interactions['ID_SKU'])

interactions_with_date = df_filtered.merge(
    interactions[['Телефон_new', 'ID_SKU', 'user_id', 'item_id', 'count']],
    on=['Телефон_new', 'ID_SKU'], how='inner'
).sort_values('Дата')

split_date = interactions_with_date['Дата'].quantile(0.8)
print(f"Дата разделения: {split_date}")

train_data = interactions_with_date[interactions_with_date['Дата'] < split_date]
test_data = interactions_with_date[interactions_with_date['Дата'] >= split_date]

train_interactions = train_data.groupby(['user_id', 'item_id'])['count'].sum().reset_index()
test_interactions = test_data.groupby(['user_id', 'item_id'])['count'].sum().reset_index()

train_users = set(train_interactions['user_id'].unique())
test_users = set(test_interactions['user_id'].unique())
print(f"Train: {len(train_interactions):,} пар, Test: {len(test_interactions):,} пар")
print(f"Cold start users: {len(test_users - train_users):,}")

Дата разделения: 2017-09-06 11:48:00
Train: 1,207,042 пар, Test: 320,341 пар
Cold start users: 16,605


In [7]:
n_users_total = len(user_encoder.classes_)
n_items_total = len(item_encoder.classes_)

def build_sparse(df, n_users, n_items):
    return csr_matrix(
        (np.ones(len(df)),
         (df['user_id'].values, df['item_id'].values)),
        shape=(n_users, n_items)
    )

train_matrix = build_sparse(train_interactions, n_users_total, n_items_total)
test_matrix = build_sparse(test_interactions, n_users_total, n_items_total)

print(f"Train matrix: {train_matrix.shape}, ненулевых: {train_matrix.nnz:,}")
print(f"Test matrix:  {test_matrix.shape}, ненулевых: {test_matrix.nnz:,}")

Train matrix: (129421, 71502), ненулевых: 1,207,042
Test matrix:  (129421, 71502), ненулевых: 320,341


In [ ]:
model = LightFM(
    no_components=64,
    loss='warp',
    learning_rate=0.05,
    item_alpha=1e-6,
    user_alpha=1e-6,
    random_state=42
)

NUM_EPOCHS = 20
NUM_THREADS = 16

train_precisions = []

from lightfm.evaluation import precision_at_k as lfm_precision_at_k

for epoch in range(NUM_EPOCHS):
    model.fit_partial(train_matrix, epochs=1, num_threads=NUM_THREADS)
    p = lfm_precision_at_k(model, train_matrix, k=10, num_threads=NUM_THREADS).mean()
    train_precisions.append(p)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: train Precision@10 = {p:.4f}")

Epoch 1/20: train Precision@10 = 0.0381


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, NUM_EPOCHS + 1), train_precisions, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Precision@10 (train)')
plt.title('LightFM WARP — кривая обучения')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def get_recommendations_lfm(model, user_id, train_matrix, n_items, k=20):
    """Top-k рекомендаций для пользователя, исключая уже купленные."""
    scores = model.predict(user_id, np.arange(n_items))
    bought = set(train_matrix[user_id].indices)
    candidate_items = [i for i in np.argsort(-scores) if i not in bought]
    return candidate_items[:k]


def precision_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(rec_k) if rec_k else 0.0

def recall_at_k(recommended, relevant, k):
    rel = set(relevant)
    return len(set(recommended[:k]) & rel) / len(rel) if rel else 0.0

def map_at_k(recommended, relevant, k):
    rel = set(relevant)
    if not rel:
        return 0.0
    score, hits = 0.0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in rel:
            hits += 1.0
            score += hits / (i + 1.0)
    return score / min(len(rel), k)

def ndcg_at_k(recommended, relevant, k):
    rel = set(relevant)
    if not rel:
        return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in rel)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(rel), k)))
    return dcg / idcg if idcg > 0 else 0.0

In [ ]:
def evaluate_model(model, train_matrix, test_interactions, n_items, k_values=[5, 10, 20]):
    test_user_items = test_interactions.groupby('user_id')['item_id'].apply(list).to_dict()
    train_users = set(np.where(train_matrix.getnnz(axis=1) > 0)[0])
    eval_users = [u for u in test_user_items if u in train_users]

    print(f"Оценка на {len(eval_users):,} пользователях...")

    metrics = {k: {'precision': [], 'recall': [], 'map': [], 'ndcg': []} for k in k_values}

    for user_id in eval_users:
        rec_items = get_recommendations_lfm(model, user_id, train_matrix, n_items, k=max(k_values))
        relevant = test_user_items[user_id]
        for k in k_values:
            metrics[k]['precision'].append(precision_at_k(rec_items, relevant, k))
            metrics[k]['recall'].append(recall_at_k(rec_items, relevant, k))
            metrics[k]['map'].append(map_at_k(rec_items, relevant, k))
            metrics[k]['ndcg'].append(ndcg_at_k(rec_items, relevant, k))

    return {
        k: {m: float(np.mean(v)) for m, v in mv.items()}
        for k, mv in metrics.items()
    }

In [ ]:
results = evaluate_model(model, train_matrix, test_interactions, n_items_total)

In [ ]:
print("=== LightFM WARP Metrics (Full Year) ===")
for k in [5, 10, 20]:
    r = results[k]
    print(f"\nK={k}:")
    print(f"  Precision@{k}: {r['precision']:.4f}")
    print(f"  Recall@{k}:    {r['recall']:.4f}")
    print(f"  MAP@{k}:       {r['map']:.4f}")
    print(f"  NDCG@{k}:      {r['ndcg']:.4f}")

In [ ]:
metrics_df = pd.DataFrame(results).T
metrics_df.index.name = 'K'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(metrics_df.index, metrics_df['precision'], marker='o', linewidth=2)
axes[0, 0].set_xlabel('K'); axes[0, 0].set_ylabel('Precision@K')
axes[0, 0].set_title('Precision@K'); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(metrics_df.index, metrics_df['recall'], marker='o', linewidth=2, color='orange')
axes[0, 1].set_xlabel('K'); axes[0, 1].set_ylabel('Recall@K')
axes[0, 1].set_title('Recall@K'); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(metrics_df.index, metrics_df['map'], marker='o', linewidth=2, color='green')
axes[1, 0].set_xlabel('K'); axes[1, 0].set_ylabel('MAP@K')
axes[1, 0].set_title('MAP@K'); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(metrics_df.index, metrics_df['ndcg'], marker='o', linewidth=2, color='red')
axes[1, 1].set_xlabel('K'); axes[1, 1].set_ylabel('NDCG@K')
axes[1, 1].set_title('NDCG@K'); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()